Step 1: Import Required Libraries



In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from nltk.tokenize import word_tokenize
import nltk

# Download required NLTK tokenizer
nltk.download('punkt')


Step 2: Load the Dataset

In [ ]:
# Load test data with UTF-16 encoding
test_data_path = '/content/test_dataset.txt'

with open(test_data_path, 'r', encoding='utf-16') as file:
    sentences = [line.strip().split(" ", 1) for line in file.readlines()]

# Process the sentences and load into DataFrame
import pandas as pd

df = pd.DataFrame(sentences, columns=["label", "sentence"])
df["label"] = df["label"].astype(int)

# Check the first few rows of the dataframe to ensure it's loaded correctly
df.head()

Step 3: Handle Missing Values


In [ ]:
# Check for missing values and handle them
df['sentence'] = df['sentence'].fillna('')  # Replace NaN with an empty string

# Optional: Drop rows with missing sentences if needed
# df = df.dropna(subset=['sentence'])

print("Missing values handled. Dataset preview:")
print(df.isnull().sum())  # Verify no missing values remain


Step 4: Split Data into Features and Labels


In [ ]:
# Split data into sentences (features) and labels (target)
X = df['sentence']  # Features: sentences
y = df['label']     # Labels: 1 (correct), 0 (incorrect)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train-test split completed.")
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")


Step 5: Text Vectorization Using CountVectorizer



In [ ]:
!pip install nltk
import nltk

# Download the required NLTK data package
nltk.download('punkt_tab')

# Initialize CountVectorizer with a custom tokenizer
vectorizer = CountVectorizer(tokenizer=word_tokenize, token_pattern=None)

# Convert text data into numeric vectors
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print("Vectorization completed. Feature matrix size:")
print(f"Training data: {X_train_vectorized.shape}, Testing data: {X_test_vectorized.shape}")

Step 6: Train the Random Forest Classifier


In [ ]:
# Initialize the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the vectorized training data
model.fit(X_train_vectorized, y_train)

print("Model training completed.")


Step 7: Evaluate the Model


In [ ]:
# Predict on the test set
y_pred = model.predict(X_test_vectorized)

# Calculate and display accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")


Step 8: Predict and Correct Sentences


In [ ]:
# Define a function to predict and correct sentences
def grammar_checker(sentence):
    # Vectorize the input sentence
    vectorized_sentence = vectorizer.transform([sentence])

    # Predict whether the sentence is correct or incorrect
    prediction = model.predict(vectorized_sentence)[0]

    if prediction == 1:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Correct.")
    else:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Incorrect.")

        # Suggest a correction (mock correction here for demonstration)
        # In practice, this could involve further analysis
        print("Correct Sentence: (Correction logic required)")

# Test the function with a sample sentence
sample_sentence = "?? ????"
grammar_checker(sample_sentence)


Final Pipeline Code



In [ ]:
def grammar_checker(sentence, vectorizer, model, df):
    """
    Check the grammar of a sentence and suggest a correction if it's incorrect.

    Parameters:
    - sentence: The input sentence to check.
    - vectorizer: The trained CountVectorizer.
    - model: The trained classification model.
    - df: The original dataset (used to find corrections).
    """
    # Vectorize the input sentence
    vectorized_sentence = vectorizer.transform([sentence])

    # Predict whether the sentence is correct or incorrect
    prediction = model.predict(vectorized_sentence)[0]

    if prediction == 1:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Correct.")
    else:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Incorrect.")

        # Find a similar correct sentence from the dataset
        correct_sentences = df[df['label'] == 1]['sentence']
        most_similar = None
        max_similarity = 0

        for correct_sentence in correct_sentences:
            # Calculate similarity between input and dataset sentences
            similarity = len(set(sentence.split()).intersection(set(correct_sentence.split())))
            if similarity > max_similarity:
                max_similarity = similarity
                most_similar = correct_sentence

        if most_similar:
            print(f"Correct Sentence: {most_similar}")
        else:
            print("Correct Sentence: No suggestion found.")


In [ ]:
sentences = ["??? ??????", "?? ??????"]
for sentence in sentences:
    grammar_checker(sentence, vectorizer, model, df)
    print("\n")


Experiment with Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train_vectorized, y_train)
best_model = grid_search.best_estimator_


Evaluate Using Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_train_vectorized, y_train, cv=5, scoring='accuracy')
print(f"Cross-Validation Accuracy: {scores.mean():.2f}")


Update Grammar Checker for Accuracy

In [ ]:
def grammar_checker(sentence, vectorizer, model):
    vectorized_sentence = vectorizer.transform([sentence])
    prediction = model.predict(vectorized_sentence)[0]
    return "Correct" if prediction == 1 else "Incorrect"

sentences = ["?? ????", "?? ?????", "?? ??????"]
for sentence in sentences:
    result = grammar_checker(sentence, vectorizer, best_model)
    print(f"Sentence: {sentence} -> {result}")


Use Confusion Matrix for Insights

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_test_pred = model.predict(X_test_vectorized)
print(confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))


In [ ]:
sentences = ["??? ???", "??? ???", "??? ????"]
for sentence in sentences:
    result = grammar_checker(sentence, vectorizer, best_model)
    print(f"Sentence: {sentence} -> {result}")

In [ ]:
def grammar_checker(sentence, vectorizer, model, df):

    # Vectorize the input sentence
    vectorized_sentence = vectorizer.transform([sentence])

    # Predict whether the sentence is correct or incorrect
    prediction = model.predict(vectorized_sentence)[0]

    if prediction == 1:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Correct.")
    else:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Incorrect.")

        # Find a similar correct sentence from the dataset
        correct_sentences = df[df['label'] == 1]['sentence']
        most_similar = None
        max_similarity = 0

        for correct_sentence in correct_sentences:
            # Calculate similarity between input and dataset sentences
            # Using Jaccard similarity for better results
            set1 = set(sentence.split())
            set2 = set(correct_sentence.split())
            similarity = len(set1.intersection(set2)) / len(set1.union(set2)) if set1.union(set2) else 0

            if similarity > max_similarity:
                max_similarity = similarity
                most_similar = correct_sentence

        if most_similar:
            print(f"Correct Sentence: {most_similar}")
        else:
            print("Correct Sentence: No suggestion found.")


# Example usage (assuming you have vectorizer, model, and df defined)
# Replace with your actual vectorizer, model and DataFrame
sentences = ["??? ???", "??? ???", "??? ????"]
for sentence in sentences:
    grammar_checker(sentence, vectorizer, best_model, df)
    print("\n")

In [ ]:
sentences = ["?? ??????? ????? ???????", "??? ??????", "???? ??????? ?????" , " ?? ???? ???", "????? ???? ??????? ??????","???? ??? ??? ???? ??????"]
for sentence in sentences:
    grammar_checker(sentence, vectorizer, best_model, df)
    print("\n")

In [ ]:
import pickle

# Assuming 'model', 'vectorizer', and 'df' are defined as in the provided code

# Save the model
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save the vectorizer
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# Save the DataFrame (optional, depending on your needs)
df.to_pickle('df.pkl')



In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from nltk.tokenize import word_tokenize
import nltk
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, classification_report
import pickle
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.models import load_model
# Step 1: Import Required Libraries
#
#

# Download required NLTK tokenizer
nltk.download('punkt')
nltk.download('punkt_tab')

# Step 2: Load the Dataset
# Load test data with UTF-16 encoding
test_data_path = '/content/test_dataset.txt'

with open(test_data_path, 'r', encoding='utf-16') as file:
    sentences = [line.strip().split(" ", 1) for line in file.readlines()]

# Process the sentences and load into DataFrame

df = pd.DataFrame(sentences, columns=["label", "sentence"])
df["label"] = df["label"].astype(int)

# Check the first few rows of the dataframe to ensure it's loaded correctly
df.head()
# Step 3: Handle Missing Values
#
# Check for missing values and handle them
df['sentence'] = df['sentence'].fillna('')  # Replace NaN with an empty string

# Optional: Drop rows with missing sentences if needed
# df = df.dropna(subset=['sentence'])

print("Missing values handled. Dataset preview:")
print(df.isnull().sum())  # Verify no missing values remain

# Step 4: Split Data into Features and Labels
#
# Split data into sentences (features) and labels (target)
X = df['sentence']  # Features: sentences
y = df['label']     # Labels: 1 (correct), 0 (incorrect)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train-test split completed.")
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

# Step 5: Text Vectorization Using CountVectorizer
#
#
!pip install nltk

# Initialize CountVectorizer with a custom tokenizer
vectorizer = CountVectorizer(tokenizer=word_tokenize, token_pattern=None)

# Convert text data into numeric vectors
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print("Vectorization completed. Feature matrix size:")
print(f"Training data: {X_train_vectorized.shape}, Testing data: {X_test_vectorized.shape}")
# Step 6: Train the Random Forest Classifier
#
# Initialize the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the vectorized training data
model.fit(X_train_vectorized, y_train)

print("Model training completed.")

# Step 7: Evaluate the Model
#
# Predict on the test set
y_pred = model.predict(X_test_vectorized)

# Calculate and display accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Step 8: Predict and Correct Sentences
#
# Define a function to predict and correct sentences
def grammar_checker(sentence):
    # Vectorize the input sentence
    vectorized_sentence = vectorizer.transform([sentence])

    # Predict whether the sentence is correct or incorrect
    prediction = model.predict(vectorized_sentence)[0]

    if prediction == 1:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Correct.")
    else:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Incorrect.")

        # Suggest a correction (mock correction here for demonstration)
        # In practice, this could involve further analysis
        print("Correct Sentence: (Correction logic required)")

# Final Pipeline Code

def grammar_checker(sentence, vectorizer, model, df):
    """
    Check the grammar of a sentence and suggest a correction if it's incorrect.

    Parameters:
    - sentence: The input sentence to check.
    - vectorizer: The trained CountVectorizer.
    - model: The trained classification model.
    - df: The original dataset (used to find corrections).
    """
    # Vectorize the input sentence
    vectorized_sentence = vectorizer.transform([sentence])

    # Predict whether the sentence is correct or incorrect
    prediction = model.predict(vectorized_sentence)[0]

    if prediction == 1:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Correct.")
    else:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Incorrect.")

        # Find a similar correct sentence from the dataset
        correct_sentences = df[df['label'] == 1]['sentence']
        most_similar = None
        max_similarity = 0

        for correct_sentence in correct_sentences:
            # Calculate similarity between input and dataset sentences
            similarity = len(set(sentence.split()).intersection(set(correct_sentence.split())))
            if similarity > max_similarity:
                max_similarity = similarity
                most_similar = correct_sentence

        if most_similar:
            print(f"Correct Sentence: {most_similar}")
        else:
            print("Correct Sentence: No suggestion found.")

# Experiment with Hyperparameter Tuning

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train_vectorized, y_train)
best_model = grid_search.best_estimator_

# Evaluate Using Cross-Validation

scores = cross_val_score(model, X_train_vectorized, y_train, cv=5, scoring='accuracy')
print(f"Cross-Validation Accuracy: {scores.mean():.2f}")

# Update Grammar Checker for Accuracy
def grammar_checker(sentence, vectorizer, model):
    vectorized_sentence = vectorizer.transform([sentence])
    prediction = model.predict(vectorized_sentence)[0]
    return "Correct" if prediction == 1 else "Incorrect"

# Use Confusion Matrix for Insights

y_test_pred = model.predict(X_test_vectorized)
print(confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

def grammar_checker(sentence, vectorizer, model, df):

    # Vectorize the input sentence
    vectorized_sentence = vectorizer.transform([sentence])

    # Predict whether the sentence is correct or incorrect
    prediction = model.predict(vectorized_sentence)[0]

    if prediction == 1:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Correct.")
    else:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Incorrect.")




In [ ]:
def grammar_checker(sentence, vectorizer, model, df):
    # Vectorize the input sentence
    vectorized_sentence = vectorizer.transform([sentence])

    # Predict whether the sentence is correct or incorrect
    prediction = model.predict(vectorized_sentence)[0]

    if prediction == 1:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Correct.")
    else:
        print(f"Input Sentence: {sentence}")
        print("Sentence is Incorrect.")

        # Find a similar correct sentence from the dataset
        correct_sentences = df[df['label'] == 1]['sentence']
        most_similar = None
        max_similarity = 0

        for correct_sentence in correct_sentences:
            # Calculate Jaccard similarity between input and dataset sentences
            set1 = set(sentence.split())
            set2 = set(correct_sentence.split())
            similarity = len(set1.intersection(set2)) / len(set1.union(set2)) if set1.union(set2) else 0
            if similarity > max_similarity:
                max_similarity = similarity
                most_similar = correct_sentence

        if most_similar:
             print(f"Correct Sentence: {most_similar}")
        else:
            print("Correct Sentence: No suggestion found.")

while True:
    sentence = input("Enter a sentence (or type 'exit' to quit): ")
    if sentence.lower() == 'exit':
        break
    grammar_checker(sentence, vectorizer, model, df)
    print("\n")